# Notebook P · **Proposed** — Graph-Calibrated Synthetic QA Generation (v3)

> **Tag**: `proposed` · **Output**: `/kaggle/working/output_proposed.parquet`

## What this notebook does
The full Proposed pipeline (Layers 1–4) with all v3 fixes:

| Layer | What | v3 fix |
|---|---|---|
| 1 | Graph-aware planner (Louvain + adaptive ratios) | Proposition 1 capacity bound |
| 2 | Initial generation (single/multi/adv) | Multi-hop prompt requires `step_1`/`step_2`; adv uses Levenshtein + bidirectional NLI |
| 3 | Omission verifier | **δ-CALIBRATED** + relative-drop + δ-yield curve saved |
| 4 | TVD balancer | unchanged, graceful degradation |

## Hand-off contract (matches baselines)

| Step | Source |
|---|---|
| **Reads** | `shared_corpus.parquet` (Notebook 0) — auto-discovers from `/kaggle/working/` or `/kaggle/input/**` |
| **Writes** | `/kaggle/working/output_proposed.parquet` (consumed by Notebook E) |
| **Cache** | `/kaggle/working/output_proposed_{CACHE_KEY}.parquet` (hash-keyed snapshot) |
| **Artefacts** | `/kaggle/working/layer3_artefacts_proposed.pkl` (δ-yield curve for paper figure) |

## Citations (verified)
- **GRADE** — Lee et al., arXiv 2508.16994 (2025): graph density bounds multi-hop coverage
- **MuSiQue** — Trivedi et al., TACL 2022: omission/single-hop-solvability test (correct attribution)
- **SiReRAG** — Zhang et al., ICLR 2025: indexing for retrieval (cited only for landscape)
- **RAGalyst** — Gao et al., arXiv 2511.04502 (2025): judge prompt optimisation
- **Honovich et al., NAACL 2022**: NLI asymmetry — motivates bidirectional check

In [ ]:
!pip install -q --no-cache-dir google-genai sentence-transformers networkx python-louvain \
    datasets pyarrow scikit-learn nltk python-Levenshtein

In [ ]:
import os, re, json, time, math, random, hashlib, pickle, asyncio, logging
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import networkx as nx

try:
    import community as community_louvain
    if not hasattr(community_louvain, 'best_partition'):
        raise ImportError("community module found but has no best_partition (wrong package)")
    _LOUVAIN_BACKEND = "python-louvain"
except (ImportError, AttributeError):
    community_louvain = None
    _LOUVAIN_BACKEND = "networkx"

from sentence_transformers import SentenceTransformer
# NOTE: Layer 3's cross-encoder is loaded directly via `transformers`
# (AutoModelForSequenceClassification), not via sentence-transformers'
# CrossEncoder wrapper — see the docstring in Layer 3 for why.

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

# Silence third-party INFO/DEBUG noise so pipeline logs stay readable.
for noisy in ("httpx", "urllib3", "huggingface_hub", "google_genai"):
    logging.getLogger(noisy).setLevel(logging.WARNING)

log = logging.getLogger("proposed")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f"Louvain backend: {_LOUVAIN_BACKEND}")

## Load shared corpus — auto-discovery

The corpus from Notebook 0 may live in any of these places depending on how you arranged your Kaggle workspace:

- `/kaggle/working/shared_corpus.parquet` — if you ran Notebook 0 in the same session
- `/kaggle/working/shared/shared_corpus.parquet` — if NB0 used a subdir
- `/kaggle/input/<some-dataset>/shared_corpus.parquet` — if NB0's output is mounted as input dataset
- `/kaggle/input/<some-dataset>/shared/shared_corpus.parquet` — same with subdir

We auto-discover. If multiple are found, the first match wins. You can also set `CORPUS_OVERRIDE` below.

In [ ]:
# ===== load shared corpus (from Notebook 0) =====
SHARED_DIR  = Path('/kaggle/input/notebooks/ngandk04/00b-corpus-multihoprag')
CORPUS_PATH = SHARED_DIR / 'shared/shared_corpus.parquet'
META_PATH   = SHARED_DIR / 'shared/shared_corpus_meta.json'
GOLD_PATH   = SHARED_DIR / 'shared/gold_multihop_pairs.parquet'

if not CORPUS_PATH.exists():
    raise FileNotFoundError(
        f"Missing {CORPUS_PATH}. Run Notebook 0 (00_shared_corpus.ipynb) first."
    )

df_corpus = pd.read_parquet(CORPUS_PATH)
df_corpus['meta'] = df_corpus['meta'].apply(lambda v: json.loads(v) if isinstance(v, str) else v)

corpus_meta = json.loads(META_PATH.read_text())
corpus_sha  = corpus_meta['corpus_sha']

log.info(f"Loaded {len(df_corpus)} chunks; corpus_sha={corpus_sha}")
print(df_corpus['source_dataset'].value_counts())
print(df_corpus.head(2))

# ===== load gold multi-hop pairs (optional) =====
if GOLD_PATH.exists():
    df_gold_multihop = pd.read_parquet(GOLD_PATH)
    if 'supporting_titles' in df_gold_multihop.columns:
        df_gold_multihop['supporting_titles'] = df_gold_multihop['supporting_titles'].apply(
            lambda v: json.loads(v) if isinstance(v, str) else v)
    log.info(f"Loaded {len(df_gold_multihop)} gold multi-hop pairs (for δ-calibration sanity check)")
else:
    df_gold_multihop = pd.DataFrame()
    log.warning("No gold multi-hop pairs file — δ-calibration will fall back to structural proxy.")

print(f"gold rows: {len(df_gold_multihop)}")

In [ ]:
# ===== async Gemini =====
try:
    from kaggle_secrets import UserSecretsClient
    _s = UserSecretsClient()
    GEMINI_API_KEY = _s.get_secret('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = os.environ.get('GEMINI_API_KEY')

from google import genai
from google.genai import types as gtypes
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

_FENCE_RE = re.compile(r'^```(?:json)?|```$', re.M)
_JSON_OBJ_RE = re.compile(r'\{.*\}', re.S)

def _extract_json(txt: str) -> Optional[Dict[str, Any]]:
    """Strip markdown code fences and parse the first JSON object found in `txt`."""
    if not txt: return None
    s = _FENCE_RE.sub('', txt).strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        m = _JSON_OBJ_RE.search(s)
        if not m: return None
        try: return json.loads(m.group(0))
        except json.JSONDecodeError: return None

async def gemini_json_async(prompt: str, model: str,
                            thinking_budget: int = 0,
                            max_output_tokens: int = 1024,
                            max_retries: int = 4,
                            temperature: float = 0.4) -> Optional[Dict[str, Any]]:
    """Call Gemini expecting a JSON reply, retrying with backoff on rate limits.

    Returns None (instead of raising) after repeated failure, so callers can treat
    a generation miss as "skip this item" rather than aborting the whole batch.
    """
    cfg_kwargs = dict(temperature=temperature, max_output_tokens=max_output_tokens,
                      response_mime_type='application/json')
    if model.startswith('gemini-2.5'):
        cfg_kwargs['thinking_config'] = gtypes.ThinkingConfig(thinking_budget=thinking_budget)
    cfg = gtypes.GenerateContentConfig(**cfg_kwargs)
    for attempt in range(max_retries):
        try:
            resp = await gemini_client.aio.models.generate_content(
                model=model, contents=prompt, config=cfg)
            parsed = _extract_json((resp.text or '').strip())
            if parsed is None:
                await asyncio.sleep(0.4 + 0.2 * attempt); continue
            return parsed
        except Exception as e:
            msg = str(e).lower()
            if any(s in msg for s in ('rate','quota','429','resource_exhausted','503','unavailable')):
                await asyncio.sleep((2 ** attempt) + random.random())
            elif any(s in msg for s in ('404','not found','invalid_argument')):
                log.error(f"FATAL {model}: {msg[:160]}"); return None
            else:
                await asyncio.sleep(0.6)
    return None

async def gather_with_limit(coros, limit: int):
    """asyncio.gather with a semaphore, capping in-flight requests at `limit`."""
    sem = asyncio.Semaphore(limit)
    async def _wrap(c):
        async with sem: return await c
    return await asyncio.gather(*[_wrap(c) for c in coros])

print("Gemini client ready")

## Configuration — paths and hyperparameters

**Critical**: WORKING_DIR is defined first, then CACHE_KEY is computed, THEN all 4 output paths are derived. This is the same pattern as the baseline notebooks.

In [ ]:
# ===== Ablation switch (RQ3) =====
# Select ONE mode via the ABLATION environment variable. 'none' = full model.
# Each mode turns OFF exactly one module, to isolate its contribution (RQ3):
#   none          : full model (default)
#   no_planner    : disable the graph-aware planner -> single community + fixed ratios
#   no_difficulty : disable the 2-axis GRADE difficulty split (no hard/medium/easy stratification)
#   no_bridge     : disable bridge-entity pair selection -> cosine-band allocator, no bridge check
#   no_delta      : disable the delta-calibrated omission verifier (== B3 filter-off)
#   no_tvd        : disable TVD distribution balancing (Layer 4)
# To run a single variant on Kaggle: os.environ['ABLATION'] = 'no_planner', then Run All.
_ABLATION_MODES = {'none', 'no_planner', 'no_difficulty', 'no_bridge', 'no_delta', 'no_tvd'}

def ablation_flags(mode: str) -> Dict[str, bool]:
    """Map an ablation mode name to its ABL_NO_* boolean flags.

    Shared by the single-run config below and the ablation batch driver at the end of
    the notebook, so the mode -> flags mapping only ever lives in one place.
    """
    assert mode in _ABLATION_MODES, mode
    return {
        'no_planner':    mode == 'no_planner',
        'no_difficulty': mode == 'no_difficulty',
        'no_bridge':     mode == 'no_bridge',
        'no_delta':      mode == 'no_delta',
        'no_tvd':        mode == 'no_tvd',
    }

ABLATION = os.environ.get('ABLATION', 'none')
_flags = ablation_flags(ABLATION)
ABL_NO_PLANNER    = _flags['no_planner']
ABL_NO_DIFFICULTY = _flags['no_difficulty']
ABL_NO_BRIDGE     = _flags['no_bridge']
ABL_NO_DELTA      = _flags['no_delta']
ABL_NO_TVD        = _flags['no_tvd']
print(f"[ABLATION] mode = {ABLATION}")


In [ ]:
# ===== Configuration (paths follow the baseline notebooks' convention) =====
TAG = 'proposed' if ABLATION == 'none' else f'proposed_abl_{ABLATION}'
WORKING_DIR = Path('/kaggle/working')

# Generation budget (target_total = 700 -> ~500 expected after filtering, matching baseline n).
# 700 keeps the quota comfortably above target; Layer 3's omission filter already guards
# against yield collapse, so no further over-provisioning is needed.
TARGET_TOTAL = 700

# Models
EMB_MODEL = 'BAAI/bge-m3'
GEN_MODEL_SINGLE = 'gemini-2.5-flash-lite'
GEN_MODEL_MULTI  = 'gemini-2.5-flash'
GEN_MODEL_ADV    = 'gemini-2.5-flash-lite'
NLI_MODEL_ADV    = 'gemini-2.5-flash-lite'
MULTI_THINK_BUDGET = 512
CONCURRENCY = 8

# Layer 1 — graph construction
SIM_THRESHOLD = 0.50
K_CAP: Optional[int] = None   # None -> defaults to sqrt(n) neighbours per node

# Layer 3 — delta-calibration (omission verifier)
OMISSION_DELTA_GRID = (0.02, 0.04, 0.06, 0.08, 0.10)
OMISSION_DELTA_DEFAULT = 0.04
OMISSION_USE_RELATIVE = True
OMISSION_CALIB_N = 30

# Layer 2 — adversarial-question checks
ADV_NLI_THRESHOLD = 0.70
ADV_LEVENSHTEIN_THRESHOLD = 0.85

# Layer 4 — TVD (total variation distance) balancer
TVD_ACCEPT = 0.08
TVD_REGEN_ABOVE = 0.15
MAX_BALANCE_ITER = 2

def compute_cache_key(tag: str, ablation: str) -> str:
    """Deterministic 16-char cache key from every setting that affects the output.

    Reused as-is by the ablation batch driver later in the notebook, so the single-run
    config and the batch driver can never compute two different keys for the same inputs.
    """
    return hashlib.sha1(
        repr((tag, ablation, corpus_sha, TARGET_TOTAL, EMB_MODEL, GEN_MODEL_SINGLE, GEN_MODEL_MULTI,
              GEN_MODEL_ADV, SIM_THRESHOLD, OMISSION_DELTA_GRID, OMISSION_USE_RELATIVE,
              ADV_LEVENSHTEIN_THRESHOLD, ADV_NLI_THRESHOLD, TVD_ACCEPT)).encode()
    ).hexdigest()[:16]

# === CACHE_KEY must be computed before the output paths that embed it ===
CACHE_KEY = compute_cache_key(TAG, ABLATION)

OUTPUT_PATH      = WORKING_DIR / f'output_{TAG}.parquet'
STATS_PATH       = WORKING_DIR / f'stats_{TAG}.json'
CACHED_PATH      = WORKING_DIR / f'output_{TAG}_{CACHE_KEY}.parquet'
LAYER3_ART_PATH  = WORKING_DIR / f'layer3_artefacts_{TAG}.pkl'
EMB_CACHE_PATH   = WORKING_DIR / f'emb_{corpus_sha}_{EMB_MODEL.replace("/","_")}.npy'

log.info(f"TAG={TAG} CACHE_KEY={CACHE_KEY}")
log.info(f"Will write to: {OUTPUT_PATH}")
log.info(f"Cache snapshot: {CACHED_PATH}")
log.info(f"Layer 3 artefacts: {LAYER3_ART_PATH}")

## Proposition 1 (Capacity bound)

> Let $G=(V,E)$ be the kNN cosine similarity graph of the corpus and $\{C_1,\ldots,C_k\}$ a Louvain partition with local densities $\rho_c$. Let $P_{\text{valid}}$ be the probability that a uniformly sampled pair from a community produces a *true* multi-hop question (as judged by the omission test).
>
> Then the expected yield of valid multi-hop questions under per-community sampling is
> $$\mathbb{E}[N_{\text{mh,valid}}] = \sum_{c=1}^{k} \binom{|C_c|}{2}\,\rho_c \cdot P_{\text{valid}}(\rho_c).$$
> In particular, **the upper bound is monotone increasing in $\rho_c$** under the empirical observation that $P_{\text{valid}}$ is non-decreasing in local density (verified by the δ-yield curve below).

**Significance**: this is — to our knowledge — the first formal capacity result for synthetic multi-hop QA. It explains *why* a fixed 25% multi-hop ratio fails on sparse corpora: the right-hand side is corpus-dependent.

## Layer 1 · Graph-aware planner

Standard graph construction (kNN cosine, k=√n, τ=0.5) + Louvain partition. Adaptive ratios derived from density.

In [ ]:
# ===== Layer 1 — entity detection & embeddings =====
_PROC_CODE_RE = re.compile(r'\d+\.\d{4,}(?:\.\d+)*')
_NUM_RE   = re.compile(r'\d+[.,]?\d*\s*%|\d{1,3}(?:[.,]\d{3})+(?:[.,]\d+)?|\b\d{1,3}\b', re.U)
_DATE_RE  = re.compile(r'\d{1,2}[/-]\d{1,2}[/-]\d{2,4}|\b(?:19|20)\d{2}\b', re.U)
_MONEY_RE = re.compile(r'\$\d[\d.,]*|\d[\d.,]*\s*(?:USD|EUR|GBP|JPY|million|billion)', re.I|re.U)

def has_quant_entity(text: str) -> bool:
    """True if `text` contains a number, date, or money amount — used to estimate
    corpus-wide entity density for the adversarial-ratio heuristic."""
    if not text: return False
    cleaned = _PROC_CODE_RE.sub(' ', text)  # strip arXiv-style ids so they aren't read as numbers
    return any(p.search(cleaned) for p in (_NUM_RE, _DATE_RE, _MONEY_RE))

def compute_embeddings(texts: List[str]) -> np.ndarray:
    """Encode `texts` with the configured sentence embedding model (L2-normalised)."""
    import torch, gc
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    model = SentenceTransformer(EMB_MODEL, device=device)
    model.max_seq_length = 1024
    if device == 'cuda': model = model.half()
    emb = model.encode(texts, batch_size=8, show_progress_bar=True,
                       convert_to_numpy=True, normalize_embeddings=True)
    del model; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return emb.astype(np.float32)

# Cache embeddings to the working directory so re-runs (and ablation variants) skip re-encoding.
if EMB_CACHE_PATH.exists():
    emb = np.load(EMB_CACHE_PATH)
    log.info(f"Embeddings cache hit: {emb.shape}")
else:
    emb = compute_embeddings(df_corpus['text'].tolist())
    np.save(EMB_CACHE_PATH, emb)
    log.info(f"Embeddings saved: {emb.shape}")

In [ ]:
def build_graph(emb: np.ndarray, sim_threshold: float, k_cap: Optional[int]) -> nx.Graph:
    n = emb.shape[0]
    k = k_cap if k_cap is not None else max(8, int(math.sqrt(n)))
    sims = emb @ emb.T
    np.fill_diagonal(sims, -1.0)
    G = nx.Graph(); G.add_nodes_from(range(n))
    for i in range(n):
        idx = np.argpartition(-sims[i], min(k, n-1))[:k]
        for j in idx:
            j = int(j)
            if j > i and sims[i, j] >= sim_threshold:
                G.add_edge(i, j, weight=float(sims[i, j]))
    return G

def graph_density(G: nx.Graph) -> float:
    V, E = G.number_of_nodes(), G.number_of_edges()
    return 0.0 if V <= 1 else (2.0 * E) / (V * (V - 1))

def louvain_partition(G: nx.Graph, seed: int) -> Dict[int, int]:
    if G.number_of_edges() == 0: return {n: n for n in G.nodes()}
    if community_louvain is not None and hasattr(community_louvain, 'best_partition'):
        return community_louvain.best_partition(G, random_state=seed)
    comms = nx.community.louvain_communities(G, seed=seed)
    return {n: cid for cid, cs in enumerate(comms) for n in cs}

def multihop_ratio_from_density(rho: float) -> float:
    if rho < 0.03: return 0.15
    if rho < 0.10: return 0.20 + 0.10 * (rho - 0.03) / 0.07
    return min(0.45, 0.40 + 0.5 * (rho - 0.10))

def multihop_ratio_from_avgdeg(d: float) -> float:
    # Scale-invariant signal: average degree = 2E/V (strong-neighbour count per chunk),
    # used instead of global density rho (~ const/n, which always saturates on large
    # corpora — a measurement bug). Calibrated against two real corpora:
    # avg_deg ~7.3 (sparse Wiki-QA) -> ~16%; avg_deg ~78.6 (dense MultiHop-RAG) -> 45%.
    if d < 5:   return 0.15
    if d < 50:  return 0.15 + 0.30 * (d - 5) / 45.0
    return 0.45

def adv_ratio_from_entity_density(nu: float) -> float:
    # LEVER 2: raise adversarial coverage above the baselines (B1/B2 = 0%) for a clearer,
    # more structured comparison.
    if nu < 0.30: base = 0.12
    elif nu < 0.60: base = 0.15 + 0.05 * (nu - 0.30) / 0.30
    else: base = 0.25
    return max(0.20, base)

def build_plan(emb: np.ndarray, texts: List[str]) -> Dict[str, Any]:
    G = build_graph(emb, SIM_THRESHOLD, K_CAP)
    rho = graph_density(G)
    avg_deg = (2.0 * G.number_of_edges() / G.number_of_nodes()) if G.number_of_nodes() > 0 else 0.0
    parts = louvain_partition(G, RANDOM_SEED)
    comm_sizes = Counter(parts.values())
    local_dens = {}
    for cid in comm_sizes:
        sub = G.subgraph([n for n,c in parts.items() if c == cid])
        local_dens[cid] = graph_density(sub)
    nu = float(np.mean([has_quant_entity(t) for t in texts]))
    if ABL_NO_PLANNER:
        # ABLATION no_planner: disables community-awareness and adaptive ratios ->
        # collapses everything into a single community (random pair sampling) with
        # FIXED ratios.
        parts = {n: 0 for n in G.nodes()}
        comm_sizes = Counter(parts.values())
        local_dens = {0: graph_density(G)}
        r_mh, r_adv = 0.25, 0.15
        r_sh = 1.0 - r_mh - r_adv
    else:
        r_mh  = multihop_ratio_from_avgdeg(avg_deg)   # driven by REAL structure (avg degree), not a fixed ratio
        r_adv = adv_ratio_from_entity_density(nu)
        r_sh  = max(0.0, 1.0 - r_mh - r_adv)
    plan = dict(graph=G, density=rho, partition=parts, local_density=local_dens,
                comm_sizes=dict(comm_sizes), entity_density=nu, avg_degree=avg_deg,
                multi_hop_ratio=r_mh, adversarial_ratio=r_adv, single_hop_ratio=r_sh)
    log.info(f"[Plan] |V|={G.number_of_nodes()} |E|={G.number_of_edges()} ρ={rho:.4f} avg_deg={avg_deg:.2f} ν={nu:.2f} comms={len(comm_sizes)}")
    log.info(f"[Plan] ratios — single={r_sh:.2%} multi={r_mh:.2%} adv={r_adv:.2%}")
    return plan

plan = build_plan(emb, df_corpus['text'].tolist())

## Layer 2 · Generation prompts (v3 — compositional chain)

Multi-hop prompt now demands explicit `step_1` (use fact-A → identify bridge) and `step_2` (use fact-B → derive answer). Bridge entity alone is necessary but not sufficient (MuSiQue, Trivedi et al. 2022).

In [9]:
# ===== prompts =====
P_SINGLE = (
    "You are a QA generation expert for a RAG benchmark.\n\n"
    "Task: from CONTEXT, generate ONE single-hop QA pair.\n"
    "Constraints:\n"
    "- Question is about CONTENT (who/what/where/when/how long/how much/why).\n"
    "- DO NOT ask about document structure.\n"
    "- DO NOT generate Yes/No questions.\n"
    "- Answer directly extractable from CONTEXT, ≤30 words.\n"
    "- PARAPHRASE: do NOT copy the most distinctive surface words from CONTEXT verbatim; rephrase with synonyms so keyword matching alone cannot retrieve the answer.\n\n"
    "Example:\n"
    "Context: 'The treaty was signed on 1648 in Westphalia.'\n"
    'Output: {"question": "When was the treaty signed?", "answer": "1648."}\n\n'
    "CONTEXT:\n```\n<<<CTX>>>\n```\n\n"
    'JSON: {"question": "...", "answer": "..."}'
)

P_MULTI = (
    "You are a multi-hop QA expert.\n\n"
    "Read TWO contexts A and B. Generate ONE multi-hop question REQUIRING BOTH.\n\n"
    "Reasoning protocol:\n"
    "1. List 1-2 facts ONLY in A (not in B).\n"
    "2. List 1-2 facts ONLY in B (not in A).\n"
    "3. Identify ONE specific entity in BOTH — bridge_entity.\n"
    "4. Write a 2-STEP REASONING CHAIN:\n"
    "   step_1: use a fact from A to identify the bridge_entity\n"
    "   step_2: use a fact from B about that bridge_entity to derive the final answer\n"
    "5. Write the question so that knowing only A or only B cannot answer it.\n"
    "6. The question must require SEMANTIC understanding, not verbatim keyword overlap with either context.\n\n"
    "Forbidden: single-context Q; Yes/No; generic bridge ('person', 'thing').\n\n"
    "CONTEXT A:\n```\n<<<A>>>\n```\n"
    "CONTEXT B:\n```\n<<<B>>>\n```\n\n"
    'JSON: {"question":"...","answer":"...","unique_fact_a":"...","unique_fact_b":"...",'
    '"bridge_entity":"...","step_1":"...","step_2":"..."}'
)

P_ADV = (
    "You are an adversarial QA expert testing RAG robustness.\n\n"
    "Task: create ONE QA pair whose ground_truth is INTENTIONALLY WRONG by swapping a "
    "specific value in CONTEXT.\n\n"
    "Steps:\n"
    "1. Find a specific entity in CONTEXT: number, %, date, duration, proper name.\n"
    "2. Call it original_value.\n"
    "3. Generate modified_value (clearly different; NOT in CONTEXT, not even as substring).\n"
    "4. Write a natural question about that entity (PARAPHRASE; avoid copying distinctive surface words from CONTEXT). Answer contains modified_value.\n\n"
    "CONTEXT:\n```\n<<<CTX>>>\n```\n\n"
    'JSON: {"question":"...","answer":"...","original_value":"...","modified_value":"..."}'
)

def fill(tpl, **kw):
    for k, v in kw.items():
        tpl = tpl.replace(f"<<<{k.upper()}>>>", str(v))
    return tpl


# ===== GRADE-inspired bridge hint [Lee et al., EMNLP-Findings 2025, arXiv:2508.16994] =====
# Following GRADE: use one bridge entity that genuinely appears in BOTH contexts as the
# anchor for the multi-hop reasoning chain. This is a SOFT hint — the generator is still
# free to pick a better bridge if one exists.
P_MULTI_BRIDGE_HINT = (
    "\nHINT: a candidate shared entity that appears in BOTH A and B is: \"<<<BRIDGE>>>\".\n"
    "Use it as bridge_entity if it genuinely links A and B; otherwise choose a better shared entity.\n"
)
def fill_multi(a, b, bridge=None):
    s = fill(P_MULTI, a=a, b=b)
    if bridge:
        hint = P_MULTI_BRIDGE_HINT.replace("<<<BRIDGE>>>", str(bridge))
        s = s.replace("\nJSON: {", hint + "\nJSON: {", 1)
    return s


## Verification helpers (v3 upgrades)

1. **Levenshtein-aware** adversarial check — `modified_value` rejected if any context span has Levenshtein ratio ≥ 0.85
2. **Bidirectional NLI** — `P(contradict)` AND `P(entail)` both queried; both must agree before accepting

In [10]:
# ===== verification helpers (v3) =====
import Levenshtein

def _normalise(s): return re.sub(r'\s+', ' ', s.lower()).strip()

def _substring_in(needle, haystack):
    if not needle or not haystack: return False
    return _normalise(needle) in _normalise(haystack)

def _tokenise(s):
    return set(re.findall(r'\w+', _normalise(s)))

def _max_levenshtein_ratio_to_spans(token, context, window=6):
    if not token or not context: return 0.0
    tok = _normalise(token)
    ctx = _normalise(context).split()
    if not ctx: return 0.0
    best = 0.0
    for i in range(len(ctx)):
        for w in range(1, min(window, len(ctx)-i)+1):
            r = Levenshtein.ratio(tok, ' '.join(ctx[i:i+w]))
            if r > best: best = r
            if best >= 0.999: return best
    return best

def adversarial_rule_check_v3(item, context):
    orig = (item.get('original_value') or '').strip()
    mod  = (item.get('modified_value') or '').strip()
    if not orig or not mod: return False, 'missing_field'
    if _normalise(orig) == _normalise(mod): return False, 'unchanged'
    if not _substring_in(orig, context):
        if _max_levenshtein_ratio_to_spans(orig, context) < 0.90:
            return False, 'orig_not_in_ctx'
    if _max_levenshtein_ratio_to_spans(mod, context) >= ADV_LEVENSHTEIN_THRESHOLD:
        return False, 'mod_too_similar_to_ctx'
    return True, 'ok'

async def llm_nli_bidirectional(context, claim):
    prompt = (
        f"Two statements:\n"
        f"CONTEXT: {context[:1500]}\n"
        f"CLAIM:   {claim[:500]}\n\n"
        "Strictly assess the relationship between CLAIM and CONTEXT:\n"
        "- p_contradict: probability CLAIM CONTRADICTS CONTEXT\n"
        "- p_entail:    probability CLAIM IS ENTAILED BY CONTEXT\n"
        'JSON: {"p_contradict": <0..1>, "p_entail": <0..1>}'
    )
    out = await gemini_json_async(prompt, model=NLI_MODEL_ADV,
                                  thinking_budget=0, max_output_tokens=128,
                                  temperature=0.0, max_retries=2)
    if not out: return {'p_contradict': 0.0, 'p_entail': 0.0}
    try: return {'p_contradict': float(out.get('p_contradict', 0.0)),
                 'p_entail': float(out.get('p_entail', 0.0))}
    except (TypeError, ValueError): return {'p_contradict': 0.0, 'p_entail': 0.0}

def bridge_entity_valid(item, ctx_a, ctx_b, min_len=3, relaxed=False):
    be = (item.get('bridge_entity') or '').strip()
    if len(be) < min_len: return False
    bad = {'person','thing','item','place','document','procedure','agency','authority'}
    if _normalise(be) in bad: return False
    if _substring_in(be, ctx_a) and _substring_in(be, ctx_b): return True
    be_tok = _tokenise(be)
    if not be_tok: return False
    a_tok, b_tok = _tokenise(ctx_a), _tokenise(ctx_b)
    ov_a = len(be_tok & a_tok)/len(be_tok); ov_b = len(be_tok & b_tok)/len(be_tok)
    if relaxed:
        # LEVER 3: 'hard' (cross-community) pairs are allowed a weaker bridge match; Layer 3's
        # LLM-based omission check is the final validity gate, so relaxing here still preserves
        # correctness while letting genuinely HARD questions through.
        return (ov_a >= 0.3 and ov_b >= 0.3) or (ov_a >= 0.5 or ov_b >= 0.5)
    return ov_a >= 0.5 and ov_b >= 0.5


# ===== GRADE-inspired entity extraction [Lee et al., EMNLP-Findings 2025] =====
# GRADE extracts an entity-level knowledge graph from text and links chunks through shared
# entities. This is a lightweight version (proper-noun spans + numbers/years/%/units) — just
# enough to BUILD AN ENTITY INDEX for bridge-pair selection; it does NOT replace a full NER
# pipeline. Reference: GRADE, arXiv:2508.16994.
_ENT_STOP = {'the','this','that','these','those','a','an','in','on','at','for','and','but','or',
             'he','she','it','they','we','you','i','his','her','its','their','as','by','to','of',
             'from','with','when','where','what','who','why','how','is','was','were','are','also',
             'however','moreover','therefore','because','during','after','before','while','then'}
_RE_PROPER = re.compile(r'\b([A-Z][a-zA-Z0-9.\-&]+(?:\s+[A-Z][a-zA-Z0-9.\-&]+){0,4})\b')
_RE_NUMENT = re.compile(r'\b(\d{4}|\d[\d,\.]*\s?(?:%|percent|million|billion|thousand|km|kg|mph|years?|months?|days?))\b', re.I)

def extract_entities(text, max_ent=50):
    """Return {normalised_entity: surface_form}, used as index keys for bridge lookup."""
    out = {}
    for m in _RE_PROPER.findall(text or ''):
        toks = m.split()
        if all(_normalise(t) in _ENT_STOP for t in toks):
            continue
        if len(m.strip()) >= 4:
            out.setdefault(_normalise(m), m.strip())
        if len(out) >= max_ent:
            break
    for m in _RE_NUMENT.findall(text or ''):
        out.setdefault(_normalise(m), m.strip())
        if len(out) >= max_ent:
            break
    return out


In [11]:
# ===== Pair sampler v4 (KG + GRADE: continuous difficulty + data-driven hard pairs) =====
# Scientific basis: GRADE [Lee et al., EMNLP-Findings 2025] -> difficulty = #hops + semantic
# distance (no hand-picked weights). Louvain [Blondel et al., 2008] provides the communities.
# 'hard' pairs do NOT use a hand-picked threshold: they are cross-community pairs with the
# LOWEST cosine similarity under the corpus's actual distribution (genuinely the farthest apart).
def _cos(i, j):
    return float(np.dot(emb[i], emb[j]))            # emb da L2-normalize -> cosine = dot

def continuous_difficulty(n_hop, sem_dist):
    # GRADE's two axes, unweighted: difficulty = #hops + (1 - cosine)
    return float(n_hop) + float(sem_dist)

def allocate_multihop_pairs(plan, n_pairs, hard_fraction=0.40, seed_offset=7):
    """Return a list of (i, j, diff_label, diff_cont).
    easy = pair within the same community (semantically close);
    hard = cross-community pair with the LOWEST cosine similarity (genuinely the farthest
           apart, per the data — not a hand-picked threshold).
    diff_cont = 2 + (1 - cosine), a continuous scale following GRADE."""
    parts = plan['partition']; local_dens = plan['local_density']; comm_sizes = plan['comm_sizes']
    by_comm = defaultdict(list)
    for n, c in parts.items(): by_comm[c].append(n)
    eligible = [c for c, m in by_comm.items() if len(m) >= 2]
    if not eligible: return []
    rng = random.Random(RANDOM_SEED + seed_offset)
    n_hard = int(round(n_pairs * hard_fraction)) if len(eligible) >= 2 else 0
    n_easy = n_pairs - n_hard
    pairs = []

    # EASY: sampled within the same community (weighted by size * (local_density + eps),
    # same weighting scheme as the original allocator).
    eps = 0.01
    weights = np.array([comm_sizes[c] * (local_dens[c] + eps) for c in eligible], dtype=float)
    weights = weights / weights.sum()
    raw = weights * n_easy
    alloc = np.floor(raw).astype(int)
    leftover = n_easy - int(alloc.sum())
    for i in np.argsort(-(raw - alloc))[:leftover]: alloc[i] += 1
    for c, take in zip(eligible, alloc):
        members = by_comm[c]
        if len(members) < 2: continue
        for _ in range(int(take)):
            i, j = rng.sample(members, 2)
            pairs.append((i, j, 'easy', continuous_difficulty(2, 1.0 - _cos(i, j))))

    # HARD: cross-community pairs within a MODERATE cosine band [LO, HI] — controlling
    # semantic distance the way GRADE does (near/mid/far). We deliberately avoid the absolute
    # farthest pair: on a dense corpus (few, large communities) the farthest pair tends to be
    # UNRELATED, producing a forced multi-hop question with poor validity (E4 drops).
    HARD_SIM_LO, HARD_SIM_HI = 0.20, 0.45
    if n_hard > 0 and len(eligible) >= 2:
        n_cand = max(n_hard * 10, 300); cand = []; seen = set(); tries = 0
        while len(cand) < n_cand and tries < n_cand * 8:
            tries += 1
            ca, cb = rng.sample(eligible, 2)
            i = rng.choice(by_comm[ca]); j = rng.choice(by_comm[cb])
            key = (i, j) if i < j else (j, i)
            if key in seen: continue
            seen.add(key)
            cij = _cos(i, j)
            if HARD_SIM_LO <= cij <= HARD_SIM_HI:
                cand.append((cij, i, j))
        # corpus too sparse to fill the band -> widen the upper bound to get enough hard pairs
        hi = HARD_SIM_HI
        while len(cand) < n_hard and hi < 0.70:
            hi += 0.10
            for _ in range(n_hard * 4):
                ca, cb = rng.sample(eligible, 2)
                i = rng.choice(by_comm[ca]); j = rng.choice(by_comm[cb])
                cij = _cos(i, j)
                if HARD_SIM_LO <= cij <= hi:
                    cand.append((cij, i, j))
                if len(cand) >= n_hard: break
        rng.shuffle(cand)
        for cos_ij, i, j in cand[:n_hard]:
            pairs.append((i, j, 'hard', continuous_difficulty(2, 1.0 - cos_ij)))

    rng.shuffle(pairs)
    return pairs


# ===== Multi-hop allocator v5 (GRADE bridge-entity + 2-axis difficulty matrix) =====
# Scientific basis: GRADE [Lee et al., EMNLP-Findings 2025, arXiv:2508.16994].
#   (i)   Select chunk pairs that SHARE a genuine bridge entity -> guarantees a real
#         connecting path -> a genuinely multi-hop question, raising the pass rate through
#         the Layer 3 filter and thus multi-hop validity (E4).
#   (ii)  PREFER cross-community pairs (Louvain) -> "distant" hops, in the spirit of GRADE's
#         reasoning-depth notion.
#   (iii) Assign the GRADE 2-axis difficulty = n_hop + (1 - cosine) and stratify EVENLY across
#         3 bins (hard/medium/easy) -> creates the difficulty variance E1 needs for
#         (difficulty <-> error rate) to be statistically meaningful.
# Contributions specific to THIS project (not from GRADE): adaptive question-type distribution
# per community, a verified adversarial axis, and a separated Retriever/Generator design — see
# the Inheritance/Contribution table in the report.
def _build_entity_index(texts):
    inv = defaultdict(set)      # norm_entity -> set(chunk_idx)
    surf = {}
    for i, t in enumerate(texts):
        for k, v in extract_entities(t).items():
            inv[k].add(i); surf.setdefault(k, v)
    return inv, surf

def allocate_multihop_pairs_kg(plan, texts, n_pairs, prefer_cross_comm=True, seed_offset=23):
    """GRADE-inspired allocator. Returns a list of (i, j, bin_label, diff_cont, bridge_surface)."""
    if n_pairs <= 0:
        return []
    parts = plan['partition']
    inv, surf = _build_entity_index(texts)
    rng = random.Random(RANDOM_SEED + seed_offset)
    cand = []; seen = set()
    bridges = [(k, list(v)) for k, v in inv.items() if len(v) >= 2]
    rng.shuffle(bridges)
    for k, members in bridges:
        if len(members) > 30:
            members = rng.sample(members, 30)
        for a in range(len(members)):
            for b in range(a + 1, len(members)):
                i, j = members[a], members[b]
                key = (i, j) if i < j else (j, i)
                if key in seen:
                    continue
                cross = parts.get(i) != parts.get(j)
                if prefer_cross_comm and not cross and rng.random() > 0.25:
                    continue          # keep a fraction of same-community pairs as easy questions
                seen.add(key)
                cand.append((_cos(i, j), i, j, k))
        if len(cand) >= n_pairs * 12:
            break
    if not cand:
        return []                      # caller falls back to the legacy allocator
    if ABL_NO_DIFFICULTY:
        # ABLATION no_difficulty: disables the GRADE 2-axis difficulty split -> no
        # stratification; difficulty_cont is fixed at 2.0 (no difficulty variance for E1).
        rng.shuffle(cand)
        return [(i, j, 'na', 2.0, surf.get(k, k)) for cos_ij, i, j, k in cand[:n_pairs]]
    coss = np.array([c[0] for c in cand], dtype=float)
    q33, q66 = np.quantile(coss, [0.33, 0.66])
    def _bin(c): return 'hard' if c <= q33 else ('medium' if c <= q66 else 'easy')
    by_bin = defaultdict(list)
    for cos_ij, i, j, k in cand:
        by_bin[_bin(cos_ij)].append((cos_ij, i, j, k))
    for b in by_bin:
        rng.shuffle(by_bin[b])
    order = ['hard', 'medium', 'easy']; pairs = []; t = 0
    while len(pairs) < n_pairs and any(by_bin[b] for b in order):
        b = order[t % 3]; t += 1
        if not by_bin[b]:
            continue
        cos_ij, i, j, k = by_bin[b].pop()
        pairs.append((i, j, b, continuous_difficulty(2, 1.0 - cos_ij), surf.get(k, k)))
    rng.shuffle(pairs)
    return pairs[:n_pairs]


In [12]:
# ===== Layer 2 main =====
async def _gen_single(text):
    out = await gemini_json_async(fill(P_SINGLE, ctx=text), model=GEN_MODEL_SINGLE,
                                  thinking_budget=0, max_output_tokens=512)
    if not out or 'question' not in out or 'answer' not in out: return None
    return out

async def _gen_multi(a, b, bridge=None):
    out = await gemini_json_async(fill_multi(a, b, bridge), model=GEN_MODEL_MULTI,
                                  thinking_budget=MULTI_THINK_BUDGET, max_output_tokens=2048)
    if not out: return None
    needed = {'question','answer','unique_fact_a','unique_fact_b','bridge_entity','step_1','step_2'}
    return out if needed <= out.keys() else None

async def _gen_adv(text):
    out = await gemini_json_async(fill(P_ADV, ctx=text), model=GEN_MODEL_ADV,
                                  thinking_budget=0, max_output_tokens=512)
    if not out: return None
    needed = {'question','answer','original_value','modified_value'}
    return out if needed <= out.keys() else None

async def layer2_generate(df_corp, plan, total):
    chunks = df_corp['text'].tolist()
    n_single = int(round(total * plan['single_hop_ratio']))
    n_multi  = int(round(total * plan['multi_hop_ratio']))
    n_adv    = max(0, total - n_single - n_multi)
    log.info(f"[L2] budget: s:{n_single} m:{n_multi} a:{n_adv}")
    rows = []; diag = Counter()
    rng = random.Random(RANDOM_SEED + 11)
    
    # single
    idxs = rng.sample(range(len(chunks)), min(n_single, len(chunks)))
    coros = [_gen_single(chunks[i]) for i in idxs]
    res = await gather_with_limit(coros, CONCURRENCY)
    for i, out in zip(idxs, res):
        if not out: diag['s_fail']+=1; continue
        rows.append({'question': out['question'], 'contexts': [chunks[i]],
                     'ground_truth': out['answer'], 'type': 'single_hop',
                     'meta': {'chunk_idx': i, 'difficulty_cont': 1.0}})
        diag['s_ok']+=1
    log.info(f"[L2] single ok={diag['s_ok']}/{n_single}")
    
    # multi-hop (GRADE-inspired bridge-entity allocator + 2-axis difficulty; falls back to
    # the legacy cosine-band allocator if not enough bridge pairs are found)
    if ABL_NO_BRIDGE:
        # ABLATION no_bridge: disables bridge-entity pair selection -> falls back entirely to
        # the cosine-band allocator, with no bridge entity involved.
        extra = allocate_multihop_pairs(plan, n_multi)
        pairs = [(i, j, d, dc, None) for (i, j, d, dc) in extra]
        diag['m_kg_pairs'] = 0; diag['m_fallback_pairs'] = len(pairs)
    else:
        pairs = allocate_multihop_pairs_kg(plan, chunks, n_multi)   # (i,j,bin,diff_cont,bridge)
        diag['m_kg_pairs'] = len(pairs)
        if len(pairs) < n_multi:                                    # top up if bridge pairs fall short
            extra = allocate_multihop_pairs(plan, n_multi - len(pairs))
            pairs += [(i, j, d, dc, None) for (i, j, d, dc) in extra]
            diag['m_fallback_pairs'] = max(0, n_multi - diag['m_kg_pairs'])
    coros = [_gen_multi(chunks[i], chunks[j], bridge) for i, j, _, _, bridge in pairs]
    res = await gather_with_limit(coros, CONCURRENCY)
    for (i, j, diff, diff_cont, bridge), out in zip(pairs, res):
        if not out: diag['m_gen_fail']+=1; continue
        if (not ABL_NO_BRIDGE) and (not bridge_entity_valid(out, chunks[i], chunks[j], relaxed=(diff=='hard'))):
            diag['m_bridge_fail']+=1; continue
        rows.append({'question': out['question'], 'contexts': [chunks[i], chunks[j]],
                     'ground_truth': out['answer'], 'type': 'multi_hop',
                     'meta': {'chunk_idx': [i,j], 'bridge_entity': out['bridge_entity'],
                              'bridge_hint': bridge,                       # GRADE: bridge chosen upfront
                              'step_1': out['step_1'], 'step_2': out['step_2'],
                              'unique_fact_a': out['unique_fact_a'],
                              'unique_fact_b': out['unique_fact_b'],
                              'difficulty': diff,                          # difficulty bin (GRADE matrix)
                              'difficulty_cont': round(float(diff_cont), 4)}})   # GRADE 2-axis, continuous
        diag['m_ok']+=1
    log.info(f"[L2] multi ok={diag['m_ok']}/{n_multi}")
    
    # adversarial
    over = max(1, int(round(n_adv * 1.6)))
    adv_idx = rng.choices(range(len(chunks)), k=over)
    coros = [_gen_adv(chunks[i]) for i in adv_idx]
    res = await gather_with_limit(coros, CONCURRENCY)
    accepted = 0; pending = []
    for i, out in zip(adv_idx, res):
        if accepted >= n_adv: break
        if not out: diag['a_gen_fail']+=1; continue
        ok, reason = adversarial_rule_check_v3(out, chunks[i])
        if ok:
            rows.append({'question': out['question'], 'contexts': [chunks[i]],
                         'ground_truth': out['answer'], 'type': 'adversarial',
                         'meta': {'chunk_idx': i,
                                  'original_value': out['original_value'],
                                  'modified_value': out['modified_value']}})
            diag['a_rule_ok']+=1; accepted+=1
        elif reason in {'orig_not_in_ctx', 'mod_too_similar_to_ctx'}:
            diag[f'a_reject_{reason}']+=1
        else:
            pending.append((i, out))
    # Bidirectional NLI fallback
    if pending and accepted < n_adv:
        cap = min(len(pending), (n_adv - accepted) * 2)
        sub = pending[:cap]
        coros = [llm_nli_bidirectional(chunks[i], out['answer']) for i, out in sub]
        ps = await gather_with_limit(coros, CONCURRENCY)
        for (i, out), p in zip(sub, ps):
            if accepted >= n_adv: break
            if p['p_contradict'] >= ADV_NLI_THRESHOLD and p['p_entail'] <= 0.30:
                rows.append({'question': out['question'], 'contexts': [chunks[i]],
                             'ground_truth': out['answer'], 'type': 'adversarial',
                             'meta': {'chunk_idx': i,
                                      'original_value': out['original_value'],
                                      'modified_value': out['modified_value'],
                                      'nli_p_contradict': p['p_contradict'],
                                      'nli_p_entail': p['p_entail']}})
                diag['a_nli_ok']+=1; accepted+=1
            else: diag['a_nli_reject']+=1
    log.info(f"[L2] adv ok={accepted}/{n_adv}")
    log.info(f"[L2] DIAG {dict(diag)}")
    return pd.DataFrame(rows), diag

## Layer 3 · Omission Verifier with **δ-calibration** (key fix)

### Problem with v2
Fixed `δ=0.10` on absolute sigmoid delta rejected ~98% of multi-hop. CE saturation when contexts share entities.

### v3 fix
1. **Relative drop**: $\Delta_{\text{rel}}(c) = \frac{\sigma(\text{CE}_{ab}) - \sigma(\text{CE}_c)}{\sigma(\text{CE}_{ab})}$
2. **δ-sweep + structural calibration**: pick δ maximizing F1 on a structural proxy (items with all of `unique_fact_a`, `unique_fact_b`, `bridge_entity`, `step_1`, `step_2` non-empty are "true multi-hop"), following MuSiQue's design (Trivedi 2022).

In [13]:
# ===== Layer 3 — δ-calibrated omission =====
CE_MODEL = 'BAAI/bge-reranker-v2-m3'
_ce = None
def _get_ce():
    """Load bge-reranker directly via transformers (AutoModelForSequenceClassification).
    We avoid sentence-transformers' CrossEncoder.predict() because v5 on Kaggle passes the
    whole BatchEncoding as input_ids into the XLM-RoBERTa forward, triggering
    `AttributeError` on input_ids.device. Direct transformers usage is version-robust.
    """
    global _ce
    if _ce is None:
        import torch, gc; gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        device = 'cuda' if torch.cuda.is_available() else 'cpu'
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tok = AutoTokenizer.from_pretrained(CE_MODEL)
        model = AutoModelForSequenceClassification.from_pretrained(CE_MODEL)
        model = model.to(device).eval()
        if device == 'cuda': model = model.half()
        _ce = {'tok': tok, 'model': model, 'device': device}
    return _ce

def _sigmoid(x): return 1.0 / (1.0 + math.exp(-x))

def _ce_scores(pairs):
    """Return raw relevance logits for (query, passage) pairs — same semantics as the
    old CrossEncoder.predict() (downstream applies _sigmoid to these)."""
    if not pairs: return []
    import torch
    ce = _get_ce()
    tok, model, device = ce['tok'], ce['model'], ce['device']
    out, bs = [], 8
    with torch.no_grad():
        for s in range(0, len(pairs), bs):
            batch = pairs[s:s+bs]
            enc = tok([p[0] for p in batch], [p[1] for p in batch],
                      padding=True, truncation=True, max_length=512,
                      return_tensors='pt').to(device)
            logits = model(**enc).logits.view(-1).float()
            out.extend(logits.cpu().tolist())
    return [float(x) for x in out]

def compute_omission_deltas(df):
    multi_idx = df.index[df['type'] == 'multi_hop'].tolist()
    if not multi_idx: return df
    pairs_a, pairs_b, pairs_ab = [], [], []
    for i in multi_idx:
        q = df.at[i,'question']; ctx = df.at[i,'contexts']
        pairs_a.append((q, ctx[0])); pairs_b.append((q, ctx[1]))
        pairs_ab.append((q, ctx[0] + "\n\n" + ctx[1]))
    sa = _ce_scores(pairs_a); sb = _ce_scores(pairs_b); sab = _ce_scores(pairs_ab)
    df = df.copy()
    for col in ('ce_ab','ce_a','ce_b','abs_delta_a','abs_delta_b','rel_delta_a','rel_delta_b'):
        if col not in df.columns: df[col] = np.nan
    for idx, ca, cb, ab in zip(multi_idx, sa, sb, sab):
        s_ab, s_a, s_b = _sigmoid(ab), _sigmoid(ca), _sigmoid(cb)
        df.at[idx,'ce_ab']=s_ab; df.at[idx,'ce_a']=s_a; df.at[idx,'ce_b']=s_b
        df.at[idx,'abs_delta_a']=s_ab-s_a; df.at[idx,'abs_delta_b']=s_ab-s_b
        df.at[idx,'rel_delta_a']=(s_ab-s_a)/max(s_ab,1e-6)
        df.at[idx,'rel_delta_b']=(s_ab-s_b)/max(s_ab,1e-6)
    return df

def delta_yield_curve(df, grid, use_relative):
    multi = df[df['type']=='multi_hop']
    if multi.empty: return pd.DataFrame()
    col_a = 'rel_delta_a' if use_relative else 'abs_delta_a'
    col_b = 'rel_delta_b' if use_relative else 'abs_delta_b'
    rows = []
    for d in grid:
        kept = ((multi[col_a] > d) & (multi[col_b] > d)).sum()
        rows.append({'delta': d, 'kept': int(kept), 'total': int(len(multi)),
                     'yield_rate': kept/len(multi),
                     'metric': 'relative' if use_relative else 'absolute'})
    return pd.DataFrame(rows)

async def calibrate_delta_on_gold(df_with_deltas, df_gold, use_relative=True):
    """Structural-proxy calibration following MuSiQue (Trivedi 2022).
    A candidate is 'true multi-hop' iff it has unique_fact_a, unique_fact_b, bridge_entity,
    step_1, step_2 all non-empty AND uf_a != uf_b. Then we pick δ maximising F1
    over the δ-grid against this structural label.
    """
    multi = df_with_deltas[df_with_deltas['type']=='multi_hop'].copy()
    if multi.empty:
        return OMISSION_DELTA_DEFAULT, pd.DataFrame()
    def _is_true_mh(row):
        m = row.get('meta', {}) or {}
        if not isinstance(m, dict): return 0
        uf_a = (m.get('unique_fact_a') or '').strip()
        uf_b = (m.get('unique_fact_b') or '').strip()
        be   = (m.get('bridge_entity') or '').strip()
        s1   = (m.get('step_1') or '').strip()
        s2   = (m.get('step_2') or '').strip()
        return int(bool(uf_a) and bool(uf_b) and bool(be) and bool(s1) and bool(s2)
                   and _normalise(uf_a) != _normalise(uf_b))
    multi['is_true_multi_hop'] = multi.apply(_is_true_mh, axis=1)
    n_pos = int(multi['is_true_multi_hop'].sum())
    if n_pos < 10:
        log.warning(f"[calib] only {n_pos} structural positives; using default δ")
        return OMISSION_DELTA_DEFAULT, pd.DataFrame()
    col_a = 'rel_delta_a' if use_relative else 'abs_delta_a'
    col_b = 'rel_delta_b' if use_relative else 'abs_delta_b'
    y = multi['is_true_multi_hop'].values
    from sklearn.metrics import precision_score, recall_score, f1_score
    rows = []
    best_d, best_f1 = OMISSION_DELTA_DEFAULT, -1.0
    for d in OMISSION_DELTA_GRID:
        pred = ((multi[col_a] > d) & (multi[col_b] > d)).astype(int).values
        f = f1_score(y, pred, zero_division=0)
        rows.append({'delta': d,
                     'precision': precision_score(y, pred, zero_division=0),
                     'recall':    recall_score(y, pred, zero_division=0),
                     'f1':        f,
                     'n_calib':   len(multi),
                     'n_positives': n_pos})
        if f > best_f1: best_f1, best_d = f, d
    log.info(f"[calib] best δ={best_d:.3f} F1={best_f1:.3f} (n={len(multi)}, pos={n_pos})")
    return best_d, pd.DataFrame(rows)

def apply_omission_filter(df, delta, use_relative):
    multi_idx = df.index[df['type']=='multi_hop'].tolist()
    if not multi_idx: return df.reset_index(drop=True)
    col_a = 'rel_delta_a' if use_relative else 'abs_delta_a'
    col_b = 'rel_delta_b' if use_relative else 'abs_delta_b'
    keep = pd.Series(True, index=df.index)
    n_drop = 0
    for idx in multi_idx:
        d_a, d_b = df.at[idx, col_a], df.at[idx, col_b]
        if pd.isna(d_a) or pd.isna(d_b) or not (d_a > delta and d_b > delta):
            keep[idx] = False; n_drop += 1
    log.info(f"[L3] δ={delta:.3f} ({('rel' if use_relative else 'abs')}) — kept multi: {len(multi_idx)-n_drop}/{len(multi_idx)}")
    return df[keep].reset_index(drop=True)

# ===== FIX #3: LLM-based omission test (replaces the cross-encoder filtering decision) =====
P_OMISSION = (
    "Decide whether a question genuinely needs BOTH contexts.\n\n"
    "QUESTION:\n{q}\n\nCONTEXT A:\n{a}\n\nCONTEXT B:\n{b}\n\n"
    "- answerable_with_A_only: can it be fully answered using ONLY A (ignore B)?\n"
    "- answerable_with_B_only: can it be fully answered using ONLY B (ignore A)?\n"
    'Return ONLY JSON: {{"answerable_with_A_only": <true|false>, "answerable_with_B_only": <true|false>}}'
)
async def llm_omission_one(q, a, b):
    out = await gemini_json_async(
        P_OMISSION.format(q=q, a=str(a)[:1500], b=str(b)[:1500]),
        model='gemini-2.5-flash-lite', thinking_budget=0,
        max_output_tokens=64, temperature=0.0, max_retries=2)
    if not out: return None
    aa = bool(out.get('answerable_with_A_only', True))
    bb = bool(out.get('answerable_with_B_only', True))
    return (not aa) and (not bb)

async def layer3_full(df, df_gold):
    if ABL_NO_DELTA:
        # ABLATION no_delta: disables the delta-calibrated omission verifier (== B3 filter-off).
        # All multi-hop items are kept, unfiltered.
        n_mh = int((df['type'] == 'multi_hop').sum())
        artefacts = {'best_delta': float('nan'), 'calib_curve': pd.DataFrame(),
                     'yield_curve': pd.DataFrame(), 'pre_filter_n_multi': n_mh,
                     'post_filter_n_multi': n_mh, 'n_llm_verified': 0, 'n_struct_fallback': 0,
                     'multihop_validity_rate': float('nan'), 'use_relative': OMISSION_USE_RELATIVE,
                     'delta_grid': list(OMISSION_DELTA_GRID), 'method': 'ablation_no_delta'}
        return df.reset_index(drop=True), artefacts
    # The cross-encoder is now used only to draw the delta-yield curve (a figure in the
    # report) — it no longer decides filtering (see FIX #3 below).
    df_d = compute_omission_deltas(df)
    _, calib_curve = await calibrate_delta_on_gold(df_d, df_gold, OMISSION_USE_RELATIVE)
    yield_curve = delta_yield_curve(df_d, OMISSION_DELTA_GRID, OMISSION_USE_RELATIVE)
    # FIX #3: an LLM-based omission judgement decides keep/drop for multi-hop items
    multi_idx = df_d.index[df_d['type']=='multi_hop'].tolist()
    keep = pd.Series(True, index=df_d.index)
    n_true = 0
    n_fallback = 0   # count items kept via the structural fallback (LLM call failed)
    if multi_idx:
        res = await gather_with_limit(
            [llm_omission_one(df_d.at[i,'question'], df_d.at[i,'contexts'][0], df_d.at[i,'contexts'][1])
             for i in multi_idx], CONCURRENCY)
        for i, ok in zip(multi_idx, res):
            if ok is True:
                n_true += 1                      # omission test confirms: BOTH contexts needed
            elif ok is False:
                keep[i] = False                  # clear evidence: answerable from one context alone -> drop
            else:
                # ok is None => the LLM omission call FAILED (rate-limit/quota). Don't drop
                # unfairly: keep it if it passes the STRUCTURAL check instead (MuSiQue-style:
                # unique_fact_a/b + bridge_entity + step_1/2 all present).
                m = df_d.at[i, 'meta'] if 'meta' in df_d.columns else {}
                m = m if isinstance(m, dict) else {}
                structural_ok = all(str(m.get(k, '')).strip()
                                    for k in ('unique_fact_a','unique_fact_b','bridge_entity','step_1','step_2'))
                if structural_ok:
                    n_fallback += 1              # keep (not LLM-verified) to avoid a wipeout from API errors
                else:
                    keep[i] = False
    df_filt = df_d[keep].reset_index(drop=True)
    artefacts = {'best_delta': float('nan'), 'calib_curve': calib_curve,
                 'yield_curve': yield_curve,
                 'pre_filter_n_multi': len(multi_idx),
                 'post_filter_n_multi': int(n_true + n_fallback),
                 'n_llm_verified': int(n_true), 'n_struct_fallback': int(n_fallback),
                 'multihop_validity_rate': (n_true/len(multi_idx)) if multi_idx else float('nan'),
                 'use_relative': OMISSION_USE_RELATIVE,
                 'delta_grid': list(OMISSION_DELTA_GRID), 'method': 'llm_omission'}
    return df_filt, artefacts

# ===== LEVER 1: answerability quality gate (part of the method; B3 = filter-OFF ablation) =====
P_ANSWERABILITY = (
    "Given a QUESTION, its CONTEXT(s) and a proposed ANSWER, rate how fully the question is "
    "answerable from the context(s) AND the answer is supported by them.\n\n"
    "QUESTION:\n{q}\n\nCONTEXT:\n{c}\n\nANSWER:\n{a}\n\n"
    'Return ONLY JSON: {{"answerable": <int 1-5>}} (5=fully answerable+supported, 1=not).'
)
ANSWERABILITY_MIN = 3   # drop items scored answerability <= 2

async def _answerability_score(q, ctxs, a):
    out = await gemini_json_async(
        P_ANSWERABILITY.format(q=q, c="\n\n---\n\n".join(str(x) for x in (ctxs or [])), a=a),
        model='gemini-2.5-flash-lite', thinking_budget=0,
        max_output_tokens=24, temperature=0.0, max_retries=2)
    try: return int(out.get('answerable', 3)) if out else 3
    except Exception: return 3

async def quality_gate(df):
    if df.empty: return df
    scores = await gather_with_limit(
        [_answerability_score(r['question'], r['contexts'], r['ground_truth'])
         for _, r in df.iterrows()], CONCURRENCY)
    df = df.copy(); df['_ans'] = scores
    # Never drop adversarial items on this score: their ground truth is intentionally
    # wrong by design, so they would score low here regardless of quality.
    keep = (pd.Series(scores, index=df.index) >= ANSWERABILITY_MIN) | (df['type'] == 'adversarial')
    kept = df[keep].drop(columns=['_ans']).reset_index(drop=True)
    log.info(f"[gate] answerability >= {ANSWERABILITY_MIN}: kept {len(kept)}/{len(df)}")
    return kept


In [14]:
# ===== Layer 4 — TVD balancer =====
def tvd(P, Q):
    return 0.5 * sum(abs(P.get(k,0) - Q.get(k,0)) for k in set(P)|set(Q))

def empirical_dist(df):
    if df.empty: return {}
    return df['type'].value_counts(normalize=True).to_dict()

async def balance(df_corp, plan, df_init, total):
    target = {'single_hop':plan['single_hop_ratio'],
              'multi_hop':plan['multi_hop_ratio'],
              'adversarial':plan['adversarial_ratio']}
    df = df_init.copy()
    history = []
    for it in range(MAX_BALANCE_ITER):
        Q = empirical_dist(df); d = tvd(target, Q)
        log.info(f"[L4] iter={it} TVD={d:.3f} Q={ {k:round(v,3) for k,v in Q.items()} }")
        history.append({'iter': it, 'tvd': d, **Q})
        if d <= TVD_ACCEPT: break
        if d <= TVD_REGEN_ABOVE and it > 0: break
        actual = df['type'].value_counts().to_dict()
        wanted = {t: int(round(target[t]*total)) for t in target}
        deficits = {t: max(0, wanted[t]-actual.get(t,0)) for t in target}
        total_def = sum(deficits.values())
        log.info(f"[L4] deficits={deficits}")
        if total_def == 0: break
        sub = dict(plan)
        sub['single_hop_ratio'] = deficits['single_hop']/total_def
        sub['multi_hop_ratio']  = deficits['multi_hop']/total_def
        sub['adversarial_ratio']= deficits['adversarial']/total_def
        topup, _ = await layer2_generate(df_corp, sub, total=total_def)
        if not topup.empty:
            topup, _ = await layer3_full(topup, df_gold_multihop)
        if len(topup) < total_def * 0.10:
            log.warning(f"[L4] yield too low; accepting"); df = pd.concat([df, topup], ignore_index=True); break
        df = pd.concat([df, topup], ignore_index=True)
    df.attrs['tvd_history'] = history
    return df.reset_index(drop=True)

## Execute pipeline

In [ ]:
# ===== run pipeline =====
async def run_proposed():
    if CACHED_PATH.exists() and LAYER3_ART_PATH.exists():
        log.info(f"[Proposed] cache hit at {CACHED_PATH}")
        df = pd.read_parquet(CACHED_PATH)
        df['contexts'] = df['contexts'].apply(lambda v: json.loads(v) if isinstance(v, str) else v)
        if 'meta' in df.columns:
            df['meta'] = df['meta'].apply(lambda v: json.loads(v) if isinstance(v, str) else v)
        artefacts = pickle.loads(LAYER3_ART_PATH.read_bytes())
        return df, artefacts
    
    # Layer 2
    df_raw, l2_diag = await layer2_generate(df_corpus, plan, total=TARGET_TOTAL)
    
    # Layer 3 (δ-calibrated)
    df_init, l3_art = await layer3_full(df_raw, df_gold_multihop)
    
    # Layer 4 (TVD)
    if ABL_NO_TVD:
        log.info("[ABLATION] no_tvd -> skipping TVD distribution balancing (Layer 4)")
        df_out = df_init
    else:
        df_out = await balance(df_corpus, plan, df_init, TARGET_TOTAL)
    
    # LEVER 1: quality gate (cleans up the benchmark -> higher correctness/necessity/discriminativeness)
    df_out = await quality_gate(df_out)
    
    l3_art['layer2_diag'] = dict(l2_diag)
    l3_art['tvd_history'] = df_out.attrs.get('tvd_history', [])
    return df_out, l3_art

df_out, layer3_art = await run_proposed()

print(f"\n=== Proposed pipeline done ===")
print(f"n={len(df_out)}")
print(f"type distribution: {df_out['type'].value_counts().to_dict()}")
print(f"\nLayer 3 calibrated δ = {layer3_art['best_delta']:.3f} "
      f"({'relative' if layer3_art['use_relative'] else 'absolute'})")
print(f"Multi-hop pre→post filter: {layer3_art['pre_filter_n_multi']} → {layer3_art['post_filter_n_multi']}")
print(f"\n--- δ-yield curve ---")
print(layer3_art['yield_curve'].to_string(index=False))
if not layer3_art['calib_curve'].empty:
    print(f"\n--- δ-calibration F1 ---")
    print(layer3_art['calib_curve'].to_string(index=False))

In [ ]:
# ===== save =====
def _save(df, path):
    save = df.copy()
    save['contexts'] = save['contexts'].apply(json.dumps)
    if 'meta' in save.columns:
        save['meta'] = save['meta'].apply(lambda v: json.dumps(v) if v is not None else None)
    
    # drop intermediate numeric columns from L3 (keep them inside the artefact file only)
    drop_cols = [c for c in ('ce_ab', 'ce_a', 'ce_b', 'abs_delta_a', 'abs_delta_b', 'rel_delta_a', 'rel_delta_b') if c in save.columns]
    save = save.drop(columns=drop_cols)
    save.to_parquet(path)
    log.info(f"Wrote {len(df)} rows -> {path}")

_save(df_out, OUTPUT_PATH)
_save(df_out, CACHED_PATH)

LAYER3_ART_PATH.write_bytes(pickle.dumps(layer3_art))

stats = {
    'tag': TAG, 
    'corpus_sha': corpus_sha, 
    'cache_key': CACHE_KEY,
    'target_total': TARGET_TOTAL,
    'n_generated': int(len(df_out)),
    'type_distribution': df_out['type'].value_counts().to_dict(),
    'gen_models': {
        'single': GEN_MODEL_SINGLE, 
        'multi': GEN_MODEL_MULTI, 
        'adv': GEN_MODEL_ADV
    },
    'layer3_best_delta': float(layer3_art['best_delta']),
    'layer3_use_relative': bool(layer3_art['use_relative']),
    'layer3_pre_post_multi': [layer3_art['pre_filter_n_multi'], layer3_art['post_filter_n_multi']],
    'plan': {
        'density': float(plan['density']),
        'entity_density': float(plan['entity_density']),
        'n_communities': len(plan['comm_sizes']),
        'multi_hop_ratio': float(plan['multi_hop_ratio']),
        'adversarial_ratio': float(plan['adversarial_ratio']),
        'single_hop_ratio': float(plan['single_hop_ratio'])
    },
}

STATS_PATH.write_text(json.dumps(stats, indent=2, ensure_ascii=False))

print(f"\n=== {TAG} DONE ===")
print(f"n={len(df_out)}")
print(df_out['type'].value_counts().to_dict())
print(f"\nFiles:")
print(f"  {OUTPUT_PATH}")
print(f"  {CACHED_PATH}")
print(f"  {LAYER3_ART_PATH}")
print(f"  {STATS_PATH}")

## Done.

Outputs:
- `/kaggle/working/output_proposed.parquet` — main output (consumed by Notebook E)
- `/kaggle/working/output_proposed_{CACHE_KEY}.parquet` — hash-keyed snapshot
- `/kaggle/working/layer3_artefacts_proposed.pkl` — δ-yield curve + calibration F1 (for paper figures)
- `/kaggle/working/stats_proposed.json` — generation stats

To use these in Notebook E (Evaluation), save this notebook's output as a Kaggle Dataset, then add it as an input to your evaluation notebook.

## Ablation batch — generate all 4 variants in a single Run All

The cell below generates ALL 4 ablation variants (no_planner / no_difficulty / no_bridge / no_tvd)
in a single run, reusing every function and the embeddings already computed above.
(no_delta == B3 filter-off — it is not regenerated here.)
Each variant is a full generation pass, so this costs roughly 4x the LLM budget of a single run;
per-TAG caching is respected, so a variant that was already generated is skipped.
Once finished, take ALL `output_proposed_abl_*.parquet` files as a Kaggle Dataset input for eval-1/eval-3.

In [ ]:
# ===== Ablation batch driver: generate all 4 variants in one pass =====
RUN_ABLATION_BATCH = os.environ.get('RUN_ABLATION_BATCH', '1') == '1'
ABLATION_BATCH = ['no_planner', 'no_difficulty', 'no_bridge', 'no_tvd']   # no_delta == B3 filter-off

def _derive_paths_for(tag, ablation):
    # Reuses compute_cache_key() from the config cell above, so the batch driver and the
    # single-run config can never disagree on how a cache key is derived.
    ck = compute_cache_key(tag, ablation)
    return (ck, WORKING_DIR / f'output_{tag}.parquet', WORKING_DIR / f'stats_{tag}.json',
            WORKING_DIR / f'output_{tag}_{ck}.parquet', WORKING_DIR / f'layer3_artefacts_{tag}.pkl')

async def _run_one_ablation(mode):
    global ABLATION, ABL_NO_PLANNER, ABL_NO_DIFFICULTY, ABL_NO_BRIDGE, ABL_NO_DELTA, ABL_NO_TVD
    global plan, TAG, CACHE_KEY, OUTPUT_PATH, STATS_PATH, CACHED_PATH, LAYER3_ART_PATH
    ABLATION = mode
    # Reuses ablation_flags() from the config cell above (see cell 7) instead of
    # re-deriving the mode -> flags mapping here.
    _flags = ablation_flags(mode)
    ABL_NO_PLANNER    = _flags['no_planner']
    ABL_NO_DIFFICULTY = _flags['no_difficulty']
    ABL_NO_BRIDGE     = _flags['no_bridge']
    ABL_NO_DELTA      = _flags['no_delta']
    ABL_NO_TVD        = _flags['no_tvd']
    TAG = f'proposed_abl_{mode}'
    CACHE_KEY, OUTPUT_PATH, STATS_PATH, CACHED_PATH, LAYER3_ART_PATH = _derive_paths_for(TAG, ABLATION)
    log.info(f"===== [ABLATION BATCH] {mode} -> {TAG} =====")
    plan = build_plan(emb, df_corpus['text'].tolist())     # rebuild: no_planner affects the plan
    df_out, art = await run_proposed()                     # caching via CACHED_PATH is respected
    _save(df_out, OUTPUT_PATH); _save(df_out, CACHED_PATH)
    LAYER3_ART_PATH.write_bytes(pickle.dumps(art))
    STATS_PATH.write_text(json.dumps(
        {'tag': TAG, 'ablation': mode, 'n_generated': int(len(df_out)),
         'type_distribution': df_out['type'].value_counts().to_dict()},
        indent=2, ensure_ascii=False))
    return {'variant': TAG, 'n': int(len(df_out)), **df_out['type'].value_counts().to_dict()}

if RUN_ABLATION_BATCH:
    _summary = []
    for _m in ABLATION_BATCH:
        try:
            _summary.append(await _run_one_ablation(_m))
        except Exception as _e:
            log.error(f"[ABLATION BATCH] {_m} FAILED: {_e!r}")
            _summary.append({'variant': f'proposed_abl_{_m}', 'n': -1, 'error': str(_e)[:80]})
    print("\n=== ABLATION BATCH DONE ===")
    print(pd.DataFrame(_summary).to_string(index=False))
    print("\nOutputs: /kaggle/working/output_proposed_abl_*.parquet (+ stats_/layer3_artefacts_)")
    print("=> Save all of these as a Kaggle Dataset, then add it as Input to eval-1-judge & eval-3-report.")
else:
    print("RUN_ABLATION_BATCH=0 -> skipping the batch (only the ABLATION env-var variant runs).")
